In [1]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

BASE_PATH   = Path(r"C:\Users\ian.forbes\FinalProject")
OUTPUT_PATH = BASE_PATH / "eda_outputs"
OUTPUT_PATH.mkdir(exist_ok=True)

GROUP_KEYS = ['AdvertiserGroup', 'EffectiveBusinessUnit']

## Full rolling panel
I first use the full rolling panel to check whether the target and main feature relationships are reasonably consistent across observation months. I then repeat the key EDA on January 2025 because that is the snapshot used for modelling.

In [2]:
# Load the rolling observation panel

print("PART A - PANEL OVERVIEW (all observation months)")

panel = pd.read_csv(BASE_PATH / "processed" / "observation_panel.csv", low_memory=False)
panel['ObservationMonth'] = pd.to_datetime(panel['ObservationMonth'])

NUMERIC_FEATURES = [
    'commission_l12m', 'commission_avg_monthly_l12m',
    'tracking_gp_l12m', 'core_gp_l12m', 'total_fixed_gp_l12m',
    'sas_gp_l12m', 'ctf_gp_l12m',
    'etr_mean_l12m', 'fixed_fee_share_mean_l12m',
    'momentum_3m_yoy', 'momentum_6m_yoy',
    'commission_yoy_growth', 'tracking_gp_yoy_growth', 'etr_yoy_change',
    'commission_cv_l12m', 'commission_peak_concentration_l12m',
    'zero_commission_months', 'active_months_l12m',
    'consecutive_declining_months', 'negative_growth_months_l12m',
    'cpi_share_l12m', 'programme_count', 'programme_count_change',
    'programme_age_months', 'transaction_count_l12m',
]

CATEGORICAL_FEATURES = [
    'sector', 'EffectiveBusinessUnit', 'service_relationship_type',
    'service_rank', 'platform_rank',
    'former_sas_flag', 'has_cpi_flag', 'is_agency_managed',
    'partially_closed_last_12m', 'is_global_service_label',
    'has_prior_year_features', 'baseline_eligible',
]

print(f"Shape: {panel.shape}")
print(f"Observation months: {panel['ObservationMonth'].nunique()} "
      f"({panel['ObservationMonth'].min().date()} to {panel['ObservationMonth'].max().date()})")
print(f"Distinct AdvertiserGroup x BU: {panel.groupby(GROUP_KEYS).ngroups:,}")
print(f"Total rows: {len(panel):,}")

print(f"\nRows per observation month:")
print(panel.groupby('ObservationMonth').size().to_string())

missing = panel[NUMERIC_FEATURES].isnull().sum()
missing_pct = (missing / len(panel) * 100).round(1)
missing_summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(f"\nMissing values (numeric features):")
print(missing_summary[missing_summary['missing_count'] > 0].to_string())

yoy_missing = panel['commission_yoy_growth'].isnull().sum()
print(f"\nYoY features missing (expected - Jan-Dec 2024 cohort has no prior year): "
      f"{yoy_missing:,} of {len(panel):,} ({yoy_missing/len(panel)*100:.1f}%)")

PART A - PANEL OVERVIEW (all observation months)
Shape: (264750, 72)
Observation months: 18 (2024-01-01 to 2025-06-01)
Distinct AdvertiserGroup x BU: 20,758
Total rows: 264,750

Rows per observation month:
ObservationMonth
2024-01-01    13653
2024-02-01    13730
2024-03-01    13851
2024-04-01    13996
2024-05-01    14216
2024-06-01    14382
2024-07-01    14595
2024-08-01    14816
2024-09-01    14951
2024-10-01    14982
2024-11-01    15053
2024-12-01    15165
2025-01-01    15247
2025-02-01    15255
2025-03-01    15195
2025-04-01    15198
2025-05-01    15283
2025-06-01    15182

Missing values (numeric features):
                                    missing_count  missing_pct
etr_mean_l12m                               20373       7.7000
fixed_fee_share_mean_l12m                   14785       5.6000
momentum_3m_yoy                            200612      75.8000
momentum_6m_yoy                            198694      75.0000
commission_yoy_growth                      197493      74.6000
tra

In [3]:

print("PART B - TARGET VARIABLE ANALYSIS (all observation months)")

print(f"\nClass balance (primary 10% threshold):")
print(panel['target_decline'].value_counts())
print(f"Decline rate: {panel['target_decline'].mean()*100:.1f}%")

print(f"\nSensitivity across thresholds:")
for thresh in [5, 10, 15]:
    col = f'target_decline_{thresh}pct'
    dr  = panel[col].mean() * 100
    print(f"  {thresh}%: Decline={panel[col].sum():,} ({dr:.1f}%)  "
          f"Stable/Growing={len(panel)-panel[col].sum():,} ({100-dr:.1f}%)")

monthly_balance = panel.groupby('ObservationMonth')['target_decline'].agg(
    total='count', decline='sum'
)
monthly_balance['decline_rate_pct'] = (
    monthly_balance['decline'] / monthly_balance['total'] * 100
).round(1)
print(f"\nClass balance by observation month:")
print(monthly_balance.to_string())

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(monthly_balance.index.astype(str), monthly_balance['decline_rate_pct'], color='darkgray')
ax.axhline(50, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
ax.set_title('Decline Rate by Observation Month (10% threshold)', fontsize=13, fontweight='bold')
ax.set_xlabel('Observation Month')
ax.set_ylabel('Decline Rate (%)')
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "panel_B1_decline_rate_by_month.png", dpi=150)
plt.close()

print(f"\nDecline severity (target_decline_pct) for Decline = 1 rows:")
print(panel[panel['target_decline'] == 1]['target_decline_pct'].describe())

fig, ax = plt.subplots(figsize=(10, 4))
panel[panel['target_decline'] == 1]['target_decline_pct'].clip(-2, 2).hist(
    bins=50, ax=ax, color='darkgray', edgecolor='white'
)
ax.set_title('Distribution of Decline Severity (Decline = 1 rows)', fontsize=13, fontweight='bold')
ax.set_xlabel('Decline % (clipped at ±200%)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "panel_B2_decline_severity_distribution.png", dpi=150)
plt.close()

severe_decliners = panel[panel['target_decline_pct'] >= 0.99]
print(f"\nRows with near-100% decline: {len(severe_decliners):,} "
      f"({len(severe_decliners)/len(panel)*100:.1f}%)")
print(f"As % of all Decline rows: {len(severe_decliners)/panel['target_decline'].sum()*100:.1f}%")
print(f"\nService relationship breakdown:")
print(severe_decliners['service_relationship_type'].value_counts(normalize=True).mul(100).round(1))
print(f"\nSector breakdown:")
print(severe_decliners['sector'].value_counts(normalize=True).mul(100).round(1))
print(f"\nFormer SAS flag:")
print(severe_decliners['former_sas_flag'].value_counts(normalize=True).mul(100).round(1))

PART B - TARGET VARIABLE ANALYSIS (all observation months)

Class balance (primary 10% threshold):
target_decline
0.0000    152705
1.0000    112045
Name: count, dtype: int64
Decline rate: 42.3%

Sensitivity across thresholds:
  5%: Decline=118,641.0 (44.8%)  Stable/Growing=146,109.0 (55.2%)
  10%: Decline=112,045.0 (42.3%)  Stable/Growing=152,705.0 (57.7%)
  15%: Decline=105,226.0 (39.7%)  Stable/Growing=159,524.0 (60.3%)

Class balance by observation month:
                  total    decline  decline_rate_pct
ObservationMonth                                    
2024-01-01        13653 6,013.0000           44.0000
2024-02-01        13730 6,103.0000           44.5000
2024-03-01        13851 6,198.0000           44.7000
2024-04-01        13996 6,255.0000           44.7000
2024-05-01        14216 6,284.0000           44.2000
2024-06-01        14382 6,312.0000           43.9000
2024-07-01        14595 6,430.0000           44.1000
2024-08-01        14816 6,463.0000           43.6000
2024-09

In [4]:

print("PART C - FEATURE DISTRIBUTIONS (all observation months)")

decline_0 = panel[panel['target_decline'] == 0]
decline_1 = panel[panel['target_decline'] == 1]

summary_cols = [
    'commission_l12m', 'commission_avg_monthly_l12m',
    'etr_mean_l12m', 'fixed_fee_share_mean_l12m',
    'momentum_3m_yoy', 'momentum_6m_yoy',
    'commission_yoy_growth', 'consecutive_declining_months',
    'negative_growth_months_l12m', 'zero_commission_months',
    'cpi_share_l12m', 'ctf_gp_l12m', 'total_fixed_gp_l12m',
    'programme_count', 'programme_age_months',
]
comparison = pd.DataFrame({
    'Stable/Growing': decline_0[summary_cols].mean(),
    'Decline':        decline_1[summary_cols].mean(),
})
comparison['Difference'] = comparison['Decline'] - comparison['Stable/Growing']
comparison['Pct_Diff']   = (
    comparison['Difference'] / comparison['Stable/Growing'].abs() * 100
).round(1)
print("\nKey feature means by target class:")
print(comparison.to_string())

key_numeric = [
    'commission_l12m', 'momentum_3m_yoy',
    'etr_mean_l12m', 'fixed_fee_share_mean_l12m',
    'consecutive_declining_months', 'negative_growth_months_l12m',
    'commission_cv_l12m', 'programme_age_months',
]

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()
for i, feat in enumerate(key_numeric):
    data = panel[feat].dropna()
    clip_low, clip_high = data.quantile(0.01), data.quantile(0.99)
    d0 = decline_0[feat].clip(clip_low, clip_high).dropna()
    d1 = decline_1[feat].clip(clip_low, clip_high).dropna()
    axes[i].hist(d0, bins=40, alpha=0.6, color='lightblue', label='Stable/Growing', density=True)
    axes[i].hist(d1, bins=40, alpha=0.6, color='red', label='Decline',        density=True)
    axes[i].set_title(feat, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Density')
    axes[i].legend(fontsize=8)
plt.suptitle('Feature Distributions by Target Class (all observation months)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "panel_C1_feature_distributions.png", dpi=150, bbox_inches='tight')
plt.close()

print(f"\nCategorical feature breakdown by target:")
for feat in ['sector', 'service_relationship_type', 'EffectiveBusinessUnit',
             'former_sas_flag', 'has_cpi_flag', 'partially_closed_last_12m']:
    ct = pd.crosstab(panel[feat], panel['target_decline'], normalize='index') * 100
    ct.columns = ['Stable/Growing_%', 'Decline_%']
    ct['Total_Rows'] = panel.groupby(feat)['target_decline'].count()
    print(f"\n--- {feat} ---")
    print(ct.sort_values('Decline_%', ascending=False).to_string())

PART C - FEATURE DISTRIBUTIONS (all observation months)

Key feature means by target class:
                              Stable/Growing     Decline   Difference  Pct_Diff
commission_l12m                  75,579.6216 60,926.5585 -14,653.0631  -19.4000
commission_avg_monthly_l12m       6,298.3018  5,077.2132  -1,221.0886  -19.4000
etr_mean_l12m                         1.5972      0.8663      -0.7309  -45.8000
fixed_fee_share_mean_l12m             0.1643      0.0720      -0.0923  -56.2000
momentum_3m_yoy                      33.2316      4.3451     -28.8865  -86.9000
momentum_6m_yoy                      66.2578     10.2193     -56.0385  -84.6000
commission_yoy_growth               168.9622     17.6556    -151.3066  -89.6000
consecutive_declining_months          0.5689      0.8676       0.2987   52.5000
negative_growth_months_l12m           3.4186      4.9973       1.5786   46.2000
zero_commission_months                1.6900      1.1407      -0.5493  -32.5000
cpi_share_l12m              

In [5]:

print("PART D - COMMERCIAL HYPOTHESIS TESTING (all observation months)")

def fixed_fee_band(x):
    if x == 0:          return '0 - No fixed fees'
    elif x < 0.10:      return '1 - Low (<10%)'
    elif x < 0.30:      return '2 - Medium (10-30%)'
    elif x < 0.60:      return '3 - High (30-60%)'
    else:               return '4 - Very high (>60%)'

print("\nD.1 Fixed fee share vs decline rate:")
panel['fixed_fee_band'] = panel['fixed_fee_share_mean_l12m'].fillna(0).apply(fixed_fee_band)
ff_analysis = panel.groupby('fixed_fee_band').agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
    avg_etr=('etr_mean_l12m', 'mean'),
    avg_commission=('commission_l12m', 'mean'),
).assign(decline_rate=lambda x: x['decline_count'] / x['total'] * 100)
print(ff_analysis.to_string())

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(ff_analysis.index, ff_analysis['decline_rate'], color='darkgray')
ax.set_title('Decline Rate by Fixed Fee Share of GP', fontsize=13, fontweight='bold')
ax.set_xlabel('Fixed Fee Share Band')
ax.set_ylabel('Decline Rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "panel_D1_fixed_fee_share_vs_decline.png", dpi=150)
plt.close()

print("\nD.2 ETR YoY change vs decline (baseline-eligible rows only):")
baseline_rows = panel[panel['baseline_eligible'] == 1].copy()
baseline_rows['etr_direction'] = baseline_rows['etr_yoy_change'].apply(
    lambda x: 'Rising' if x > 0 else ('Falling' if x < 0 else 'Flat') if pd.notna(x) else 'Unknown'
)
etr_analysis = baseline_rows.groupby('etr_direction')['target_decline'].agg(
    total='count', decline='sum'
).assign(decline_rate=lambda x: x['decline'] / x['total'] * 100)
print(etr_analysis.to_string())

print("\nD.3 Former SAS flag vs decline rate:")
sas_analysis = panel.groupby('former_sas_flag').agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
    avg_commission=('commission_l12m', 'mean'),
    avg_sas_gp=('sas_gp_l12m', 'mean'),
).assign(decline_rate=lambda x: x['decline_count'] / x['total'] * 100)
print(sas_analysis.to_string())

print("\nD.3b Former SAS x fixed fee share cross-tab:")
panel['has_fixed_fees'] = (panel['total_fixed_gp_l12m'] > 0).astype(int)
sas_ff_ct = pd.crosstab(
    panel['former_sas_flag'], panel['has_fixed_fees'],
    values=panel['target_decline'], aggfunc='mean'
) * 100
sas_ff_ct.index = ['Non-SAS', 'Former SAS']
sas_ff_ct.columns = ['No Fixed Fees', 'Has Fixed Fees']
print(sas_ff_ct.round(1).to_string())

print("\nD.4 Service relationship type vs decline rate:")
svc_analysis = panel.groupby('service_relationship_type').agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
    avg_commission=('commission_l12m', 'mean'),
).assign(
    decline_rate=lambda x: x['decline_count'] / x['total'] * 100,
    pct_of_advertisers=lambda x: x['total'] / x['total'].sum() * 100,
)
print(svc_analysis.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(svc_analysis.index, svc_analysis['decline_rate'], color='darkgray')
axes[0].set_title('Decline Rate by Service Relationship', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Decline Rate (%)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].bar(svc_analysis.index, svc_analysis['avg_commission'] / 1000, color='green')
axes[1].set_title('Avg Commission L12M by Service Relationship (€k)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Avg Commission (€k)')
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "panel_D4_service_relationship_analysis.png", dpi=150)
plt.close()

print("\nD.5 Sector breakdown:")
sector_analysis = panel.groupby('sector').agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
    avg_commission=('commission_l12m', 'mean'),
).assign(
    decline_rate=lambda x: x['decline_count'] / x['total'] * 100,
    pct_of_panel=lambda x: x['total'] / x['total'].sum() * 100,
).sort_values('total', ascending=False)
print(sector_analysis.round(1).to_string())

print("\nD.6 CPI flag vs decline rate:")
cpi_analysis = panel.groupby('has_cpi_flag').agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
    avg_cpi_share=('cpi_share_l12m', 'mean'),
).assign(decline_rate=lambda x: x['decline_count'] / x['total'] * 100)
print(cpi_analysis.to_string())

print("\nD.7 Partially closed in last 12M vs decline rate:")
pc_analysis = panel.groupby('partially_closed_last_12m').agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
).assign(decline_rate=lambda x: x['decline_count'] / x['total'] * 100)
pc_analysis.index = ['Not partially closed', 'Partially closed in last 12M']
print(pc_analysis.to_string())

print("\nD.8 3-month YoY momentum vs decline rate:")
panel['momentum_3m_band'] = pd.cut(
    panel['momentum_3m_yoy'].clip(-1, 1),
    bins=[-1, -0.2, -0.05, 0.05, 0.2, 1],
    labels=['Strong decline', 'Mild decline', 'Flat', 'Mild growth', 'Strong growth']
)
mom_analysis = panel.groupby('momentum_3m_band', observed=True).agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
).assign(decline_rate=lambda x: x['decline_count'] / x['total'] * 100)
print(mom_analysis.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(mom_analysis.index.astype(str), mom_analysis['decline_rate'], color='darkgray')
ax.set_title('Decline Rate by 3-Month YoY Commission Momentum Band', fontsize=13, fontweight='bold')
ax.set_xlabel('3-Month YoY Momentum Band')
ax.set_ylabel('Decline Rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "panel_D8_momentum_vs_decline.png", dpi=150)
plt.close()


PART D - COMMERCIAL HYPOTHESIS TESTING (all observation months)

D.1 Fixed fee share vs decline rate:
                       total  decline_count  avg_etr  avg_commission  decline_rate
fixed_fee_band                                                                    
0 - No fixed fees     210343    95,996.0000   0.2245     72,298.1856       45.6378
1 - Low (<10%)          5170     1,666.0000   0.3361    232,576.7283       32.2244
2 - Medium (10-30%)    10055     3,382.0000   0.5771    108,774.4520       33.6350
3 - High (30-60%)      12739     4,542.0000   1.4811     42,275.1521       35.6543
4 - Very high (>60%)   26443     6,459.0000  11.2681     12,320.5819       24.4261

D.2 ETR YoY change vs decline (baseline-eligible rows only):
               total     decline  decline_rate
etr_direction                                 
Falling        30207 15,086.0000       49.9421
Flat            1647    994.0000       60.3522
Rising         33372 16,844.0000       50.4735
Unknown        26134

In [6]:
# PART E - CORRELATION AND FEATURE RELATIONSHIPS (full panel)

print("PART E - CORRELATION ANALYSIS (all observation months)")

corr_with_target = (
    panel[NUMERIC_FEATURES + ['target_decline']]
    .corr()['target_decline']
    .drop('target_decline')
    .sort_values()
)
print("\nCorrelation with target_decline (sorted):")
print(corr_with_target.to_string())

fig, ax = plt.subplots(figsize=(8, 10))
colors = ['red' if v < 0 else 'lightblue' for v in corr_with_target.values]
ax.barh(corr_with_target.index, corr_with_target.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Decline Target (all observation months)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "panel_E1_correlation_with_target.png", dpi=150)
plt.close()

key_for_heatmap = [
    'target_decline',
    'commission_l12m', 'etr_mean_l12m', 'fixed_fee_share_mean_l12m',
    'momentum_3m_yoy', 'momentum_6m_yoy',
    'commission_yoy_growth', 'consecutive_declining_months',
    'negative_growth_months_l12m', 'zero_commission_months',
    'commission_cv_l12m', 'cpi_share_l12m',
    'programme_count', 'programme_age_months',
]
corr_matrix = panel[key_for_heatmap].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, annot_kws={'size': 7}
)
ax.set_title('Feature Correlation Heatmap - all observation months (lower triangle)',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "panel_E2_correlation_heatmap.png", dpi=150)
plt.close()

print("\nHighly correlated feature pairs (|r| > 0.70, excluding target):")
corr_pairs = []
feats = [f for f in key_for_heatmap if f != 'target_decline']
for i in range(len(feats)):
    for j in range(i+1, len(feats)):
        r = corr_matrix.loc[feats[i], feats[j]]
        if abs(r) > 0.70:
            corr_pairs.append({'Feature_1': feats[i], 'Feature_2': feats[j], 'Correlation': round(r, 3)})
if corr_pairs:
    print(pd.DataFrame(corr_pairs).sort_values('Correlation', ascending=False).to_string(index=False))
else:
    print("No pairs with |r| > 0.70 found.")

print("\nDecline rate by commission size quartile:")
panel['commission_quartile'] = pd.qcut(
    panel['commission_l12m'], q=4,
    labels=['Q1 (smallest)', 'Q2', 'Q3', 'Q4 (largest)']
)
gp_analysis = panel.groupby('commission_quartile', observed=True).agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
    avg_commission=('commission_l12m', 'mean'),
).assign(
    decline_rate=lambda x: x['decline_count'] / x['total'] * 100,
    pct_of_total_commission=lambda x: (
        x['avg_commission'] * x['total']
        / (x['avg_commission'] * x['total']).sum() * 100
    )
)
print(gp_analysis.round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
gp_analysis['decline_rate'].plot(kind='bar', ax=axes[0], color='darkgray')
axes[0].set_title('Decline Rate by Commission Size Quartile', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Decline Rate (%)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].tick_params(axis='x', rotation=30)
gp_analysis['pct_of_total_commission'].plot(kind='bar', ax=axes[1], color='green')
axes[1].set_title('% of Total Commission by Size Quartile', fontsize=11, fontweight='bold')
axes[1].set_ylabel('% of Commission')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "panel_E3_gp_tier_analysis.png", dpi=150)
plt.close()


PART E - CORRELATION ANALYSIS (all observation months)

Correlation with target_decline (sorted):
fixed_fee_share_mean_l12m            -0.1548
commission_cv_l12m                   -0.1116
commission_peak_concentration_l12m   -0.1042
zero_commission_months               -0.0941
total_fixed_gp_l12m                  -0.0258
programme_count                      -0.0172
etr_mean_l12m                        -0.0132
core_gp_l12m                         -0.0108
commission_avg_monthly_l12m          -0.0099
commission_l12m                      -0.0099
tracking_gp_l12m                     -0.0085
programme_count_change               -0.0083
ctf_gp_l12m                          -0.0080
momentum_3m_yoy                      -0.0072
momentum_6m_yoy                      -0.0054
cpi_share_l12m                       -0.0045
transaction_count_l12m               -0.0044
commission_yoy_growth                -0.0043
tracking_gp_yoy_growth                0.0009
etr_yoy_change                        0.0171
sa

In [7]:

# Parts F-J repeat the EDA on just the January 2025 modelling snapshot.


In [8]:

print("PART F - JAN 2025 DATASET OVERVIEW")

panel = pd.read_csv(BASE_PATH / "processed" / "modelling_dataset_jan25.csv", low_memory=False)

print(f"Feature window:    January - December 2024")
print(f"Target window:     January - December 2025")
print(f"Rows:              {len(panel):,}")
print(f"Columns:           {panel.shape[1]}")
print(f"Distinct AdvertiserGroup x BU: {panel.groupby(GROUP_KEYS).ngroups:,}")

print(f"\nYoY features available: {panel['has_prior_year_features'].sum():,} of {len(panel):,}")
print(f"Baseline eligible:       {panel['baseline_eligible'].sum():,} of {len(panel):,}")

missing = panel[NUMERIC_FEATURES].isnull().sum()
missing_pct = (missing / len(panel) * 100).round(1)
missing_summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(f"\nMissing values (numeric features):")
print(missing_summary[missing_summary['missing_count'] > 0].to_string())

PART F - JAN 2025 DATASET OVERVIEW
Feature window:    January - December 2024
Target window:     January - December 2025
Rows:              15,247
Columns:           82
Distinct AdvertiserGroup x BU: 15,247

YoY features available: 15,247 of 15,247
Baseline eligible:       15,247 of 15,247

Missing values (numeric features):
                                    missing_count  missing_pct
etr_mean_l12m                                1286       8.4000
fixed_fee_share_mean_l12m                     919       6.0000
momentum_3m_yoy                              4533      29.7000
momentum_6m_yoy                              4299      28.2000
commission_yoy_growth                        4122      27.0000
tracking_gp_yoy_growth                       4130      27.1000
etr_yoy_change                               4393      28.8000
commission_cv_l12m                           1465       9.6000
commission_peak_concentration_l12m           1286       8.4000
programme_age_months                       

In [9]:
print("PART G - JAN 2025 TARGET VARIABLE")

print(f"\nClass balance (primary 10% threshold):")
print(panel['target_decline'].value_counts())
print(f"Decline rate: {panel['target_decline'].mean()*100:.1f}%")

print(f"\nSensitivity across thresholds:")
for thresh in [5, 10, 15]:
    col = f'target_decline_{thresh}pct'
    dr  = panel[col].mean() * 100
    print(f"  {thresh}%: Decline={panel[col].sum():,} ({dr:.1f}%)  "
          f"Stable/Growing={len(panel)-panel[col].sum():,} ({100-dr:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = panel['target_decline'].value_counts().sort_index()
axes[0].bar(['Stable/Growing', 'Decline'], counts.values, color=['lightblue', 'red'])
axes[0].set_title('Class Balance (10% threshold)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontsize=10)

thresholds    = [5, 10, 15]
decline_rates = [panel[f'target_decline_{t}pct'].mean() * 100 for t in thresholds]
axes[1].bar([f'{t}%' for t in thresholds], decline_rates, color='darkgray')
axes[1].axhline(50, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
axes[1].set_title('Decline Rate by Threshold', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Decline Rate (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_ylim(0, 60)
for i, v in enumerate(decline_rates):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10)

plt.suptitle('Target Variable - January 2025 Observation Point', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "jan25_B1_class_balance.png", dpi=150)
plt.close()

print(f"\nDecline severity (target_decline_pct) for Decline = 1 rows:")
print(panel[panel['target_decline'] == 1]['target_decline_pct'].describe())

fig, ax = plt.subplots(figsize=(10, 4))
panel[panel['target_decline'] == 1]['target_decline_pct'].clip(0, 1.5).hist(
    bins=50, ax=ax, color='darkgray', edgecolor='white'
)
ax.set_title('Distribution of Decline Severity (Decline = 1 rows)', fontsize=13, fontweight='bold')
ax.set_xlabel('Decline % (clipped at 150%)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "jan25_B2_decline_severity.png", dpi=150)
plt.close()

severe = panel[panel['target_decline_pct'] >= 0.99]
print(f"\nNear-100% decline: {len(severe):,} rows "
      f"({len(severe)/panel['target_decline'].sum()*100:.1f}% of all Decline rows)")
print(f"Service relationship:\n{severe['service_relationship_type'].value_counts(normalize=True).mul(100).round(1)}")
print(f"Former SAS:\n{severe['former_sas_flag'].value_counts(normalize=True).mul(100).round(1)}")

PART G - JAN 2025 TARGET VARIABLE

Class balance (primary 10% threshold):
target_decline
0.0000    9056
1.0000    6191
Name: count, dtype: int64
Decline rate: 40.6%

Sensitivity across thresholds:
  5%: Decline=6,562.0 (43.0%)  Stable/Growing=8,685.0 (57.0%)
  10%: Decline=6,191.0 (40.6%)  Stable/Growing=9,056.0 (59.4%)
  15%: Decline=5,808.0 (38.1%)  Stable/Growing=9,439.0 (61.9%)

Decline severity (target_decline_pct) for Decline = 1 rows:
count   6,191.0000
mean        0.5402
std         0.2791
min         0.1001
25%         0.2979
50%         0.5092
75%         0.7664
max         1.0000
Name: target_decline_pct, dtype: float64

Near-100% decline: 685 rows (11.1% of all Decline rows)
Service relationship:
service_relationship_type
Self Service     84.4000
Awin Managed      8.5000
Agency Managed    7.2000
Name: proportion, dtype: float64
Former SAS:
former_sas_flag
1   69.1000
0   30.9000
Name: proportion, dtype: float64


In [10]:

print("PART H - JAN 2025 FEATURE DISTRIBUTIONS")

decline_0 = panel[panel['target_decline'] == 0]
decline_1 = panel[panel['target_decline'] == 1]

summary_cols = [
    'commission_l12m', 'commission_avg_monthly_l12m',
    'etr_mean_l12m', 'fixed_fee_share_mean_l12m',
    'momentum_3m_yoy', 'momentum_6m_yoy',
    'commission_yoy_growth', 'consecutive_declining_months',
    'negative_growth_months_l12m', 'zero_commission_months',
    'cpi_share_l12m', 'ctf_gp_l12m', 'total_fixed_gp_l12m',
    'programme_count', 'programme_age_months',
]
comparison = pd.DataFrame({
    'Stable/Growing': decline_0[summary_cols].mean(),
    'Decline':        decline_1[summary_cols].mean(),
})
comparison['Difference'] = comparison['Decline'] - comparison['Stable/Growing']
comparison['Pct_Diff']   = (
    comparison['Difference'] / comparison['Stable/Growing'].abs() * 100
).round(1)
print("\nKey feature means by target class:")
print(comparison.to_string())

key_numeric = [
    'commission_l12m', 'momentum_3m_yoy',
    'etr_mean_l12m', 'fixed_fee_share_mean_l12m',
    'consecutive_declining_months', 'negative_growth_months_l12m',
    'commission_cv_l12m', 'programme_age_months',
]

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()
for i, feat in enumerate(key_numeric):
    data = panel[feat].dropna()
    clip_low, clip_high = data.quantile(0.01), data.quantile(0.99)
    d0 = decline_0[feat].clip(clip_low, clip_high).dropna()
    d1 = decline_1[feat].clip(clip_low, clip_high).dropna()
    axes[i].hist(d0, bins=40, alpha=0.6, color='lightblue', label='Stable/Growing', density=True)
    axes[i].hist(d1, bins=40, alpha=0.6, color='red', label='Decline',        density=True)
    axes[i].set_title(feat, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Density')
    axes[i].legend(fontsize=8)
plt.suptitle('Feature Distributions by Target Class - Jan 2025', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "jan25_C1_feature_distributions.png", dpi=150, bbox_inches='tight')
plt.close()

print(f"\nCategorical feature breakdown by target:")
for feat in ['sector', 'service_relationship_type', 'EffectiveBusinessUnit',
             'former_sas_flag', 'has_cpi_flag', 'partially_closed_last_12m']:
    ct = pd.crosstab(panel[feat], panel['target_decline'], normalize='index') * 100
    ct.columns = ['Stable/Growing_%', 'Decline_%']
    ct['Total_Rows'] = panel.groupby(feat)['target_decline'].count()
    print(f"\n--- {feat} ---")
    print(ct.sort_values('Decline_%', ascending=False).to_string())

PART H - JAN 2025 FEATURE DISTRIBUTIONS

Key feature means by target class:
                              Stable/Growing     Decline  Difference  Pct_Diff
commission_l12m                  68,294.4443 67,453.5831   -840.8612   -1.2000
commission_avg_monthly_l12m       5,691.2037  5,621.1319    -70.0718   -1.2000
etr_mean_l12m                         1.6041      0.9332     -0.6709  -41.8000
fixed_fee_share_mean_l12m             0.1881      0.0991     -0.0890  -47.3000
momentum_3m_yoy                       6.5998      1.3152     -5.2846  -80.1000
momentum_6m_yoy                       8.2813      2.1571     -6.1242  -74.0000
commission_yoy_growth                16.7648      3.4781    -13.2868  -79.3000
consecutive_declining_months          0.5030      0.7614      0.2584   51.4000
negative_growth_months_l12m           3.2761      4.8680      1.5920   48.6000
zero_commission_months                1.8892      1.2339     -0.6554  -34.7000
cpi_share_l12m                        0.0000      0.000

In [11]:
print("PART I - JAN 2025 COMMERCIAL HYPOTHESIS TESTING")

print("\nI.1 Fixed fee share vs decline rate:")
panel['fixed_fee_band'] = panel['fixed_fee_share_mean_l12m'].fillna(0).apply(fixed_fee_band)
ff_analysis = panel.groupby('fixed_fee_band').agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
    avg_etr=('etr_mean_l12m', 'mean'),
    avg_commission=('commission_l12m', 'mean'),
).assign(decline_rate=lambda x: x['decline_count'] / x['total'] * 100)
print(ff_analysis.to_string())

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(ff_analysis.index, ff_analysis['decline_rate'], color='darkgray')
ax.set_title('Decline Rate by Fixed Fee Share of GP - Jan 2025', fontsize=13, fontweight='bold')
ax.set_xlabel('Fixed Fee Share Band')
ax.set_ylabel('Decline Rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "jan25_D1_fixed_fee_vs_decline.png", dpi=150)
plt.close()

print("\nI.2 ETR YoY change vs decline:")
panel['etr_direction'] = panel['etr_yoy_change'].apply(
    lambda x: 'Rising' if x > 0 else ('Falling' if x < 0 else 'Flat') if pd.notna(x) else 'Unknown'
)
etr_analysis = panel.groupby('etr_direction')['target_decline'].agg(
    total='count', decline='sum'
).assign(decline_rate=lambda x: x['decline'] / x['total'] * 100)
print(etr_analysis.to_string())

print("\nI.3 Former SAS flag vs decline rate:")
sas_analysis = panel.groupby('former_sas_flag').agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
    avg_commission=('commission_l12m', 'mean'),
    avg_sas_gp=('sas_gp_l12m', 'mean'),
).assign(decline_rate=lambda x: x['decline_count'] / x['total'] * 100)
print(sas_analysis.to_string())

print("\nI.3b Former SAS and fixed fee cross-tab:")
panel['has_fixed_fees'] = (panel['total_fixed_gp_l12m'] > 0).astype(int)
sas_ff_ct = pd.crosstab(
    panel['former_sas_flag'], panel['has_fixed_fees'],
    values=panel['target_decline'], aggfunc='mean'
) * 100
sas_ff_ct.index   = ['Non-SAS', 'Former SAS']
sas_ff_ct.columns = ['No Fixed Fees', 'Has Fixed Fees']
print(sas_ff_ct.round(1).to_string())

print("\nI.4 Service relationship type vs decline rate:")
svc_analysis = panel.groupby('service_relationship_type').agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
    avg_commission=('commission_l12m', 'mean'),
).assign(
    decline_rate=lambda x: x['decline_count'] / x['total'] * 100,
    pct_of_advertisers=lambda x: x['total'] / x['total'].sum() * 100,
)
print(svc_analysis.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(svc_analysis.index, svc_analysis['decline_rate'], color='darkgray')
axes[0].set_title('Decline Rate by Service Relationship', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Decline Rate (%)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].bar(svc_analysis.index, svc_analysis['avg_commission'] / 1000, color='green')
axes[1].set_title('Avg Commission L12M by Service Relationship (€k)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Avg Commission (€k)')
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "jan25_D4_service_relationship.png", dpi=150)
plt.close()

print("\nI.5 Sector breakdown:")
sector_analysis = panel.groupby('sector').agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
    avg_commission=('commission_l12m', 'mean'),
).assign(
    decline_rate=lambda x: x['decline_count'] / x['total'] * 100,
    pct_of_panel=lambda x: x['total'] / x['total'].sum() * 100,
).sort_values('total', ascending=False)
print(sector_analysis.round(1).to_string())

print("\nI.6 CPI flag vs decline rate:")
cpi_analysis = panel.groupby('has_cpi_flag').agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
    avg_cpi_share=('cpi_share_l12m', 'mean'),
).assign(decline_rate=lambda x: x['decline_count'] / x['total'] * 100)
print(cpi_analysis.to_string())

print("\nI.7 Partial closure vs decline rate:")
pc_analysis = panel.groupby('partially_closed_last_12m').agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
).assign(decline_rate=lambda x: x['decline_count'] / x['total'] * 100)
pc_analysis.index = ['Not partially closed', 'Partially closed in last 12M']
print(pc_analysis.to_string())

print("\nI.8 3-month YoY momentum vs decline rate:")
panel['momentum_3m_band'] = pd.cut(
    panel['momentum_3m_yoy'].clip(-1, 1),
    bins=[-1, -0.2, -0.05, 0.05, 0.2, 1],
    labels=['Strong decline', 'Mild decline', 'Flat', 'Mild growth', 'Strong growth']
)
mom_analysis = panel.groupby('momentum_3m_band', observed=True).agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
).assign(decline_rate=lambda x: x['decline_count'] / x['total'] * 100)
print(mom_analysis.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(mom_analysis.index.astype(str), mom_analysis['decline_rate'], color='darkgray')
ax.set_title('Decline Rate by 3-Month YoY Momentum Band - Jan 2025',
             fontsize=13, fontweight='bold')
ax.set_xlabel('3-Month YoY Momentum Band (same quarter vs prior year)')
ax.set_ylabel('Decline Rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "jan25_D8_momentum_vs_decline.png", dpi=150)
plt.close()

print("\nI.9 YoY commission growth band vs decline rate:")
panel['yoy_band'] = pd.cut(
    panel['commission_yoy_growth'].clip(-1, 2),
    bins=[-1, -0.2, -0.05, 0.05, 0.2, 2],
    labels=['Strong decline', 'Mild decline', 'Flat', 'Mild growth', 'Strong growth']
)
yoy_analysis = panel.groupby('yoy_band', observed=True).agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
).assign(decline_rate=lambda x: x['decline_count'] / x['total'] * 100)
print(yoy_analysis.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(yoy_analysis.index.astype(str), yoy_analysis['decline_rate'], color='darkgray')
ax.set_title('Decline Rate by Full-Year YoY Commission Growth Band - Jan 2025',
             fontsize=13, fontweight='bold')
ax.set_xlabel('YoY Growth Band (2024 vs 2023)')
ax.set_ylabel('Decline Rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "jan25_D9_yoy_growth_vs_decline.png", dpi=150)
plt.close()


PART I - JAN 2025 COMMERCIAL HYPOTHESIS TESTING

I.1 Fixed fee share vs decline rate:
                      total  decline_count  avg_etr  avg_commission  decline_rate
fixed_fee_band                                                                   
0 - No fixed fees     11744     5,118.0000   0.2237     66,843.3849       43.5797
1 - Low (<10%)          251        99.0000   0.3541    296,388.0002       39.4422
2 - Medium (10-30%)     511       161.0000   0.5237    198,643.3673       31.5068
3 - High (30-60%)       802       277.0000   1.5230     71,032.9242       34.5387
4 - Very high (>60%)   1939       536.0000   8.9164      9,387.4956       27.6431

I.2 ETR YoY change vs decline:
               total    decline  decline_rate
etr_direction                                
Falling         4928 2,460.0000       49.9188
Flat             287   182.0000       63.4146
Rising          5639 2,870.0000       50.8955
Unknown         4393   679.0000       15.4564

I.3 Former SAS flag vs decline 

In [12]:
print("PART J - JAN 2025 CORRELATION ANALYSIS")

corr_with_target = (
    panel[NUMERIC_FEATURES + ['target_decline']]
    .corr()['target_decline']
    .drop('target_decline')
    .sort_values()
)
print("\nCorrelation with target_decline (sorted):")
print(corr_with_target.to_string())

fig, ax = plt.subplots(figsize=(8, 10))
colors = ['red' if v < 0 else 'lightblue' for v in corr_with_target.values]
ax.barh(corr_with_target.index, corr_with_target.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Decline Target - Jan 2025', fontsize=13, fontweight='bold')
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "jan25_E1_correlation_with_target.png", dpi=150)
plt.close()

key_for_heatmap = [
    'target_decline',
    'commission_l12m', 'etr_mean_l12m', 'fixed_fee_share_mean_l12m',
    'momentum_3m_yoy', 'momentum_6m_yoy',
    'commission_yoy_growth', 'consecutive_declining_months',
    'negative_growth_months_l12m', 'zero_commission_months',
    'commission_cv_l12m', 'cpi_share_l12m',
    'programme_count', 'programme_age_months',
]
corr_matrix = panel[key_for_heatmap].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, annot_kws={'size': 7}
)
ax.set_title('Feature Correlation Heatmap - Jan 2025', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "jan25_E2_correlation_heatmap.png", dpi=150)
plt.close()

print("\nHighly correlated feature pairs (|r| > 0.70, excluding target):")
corr_pairs = []
feats = [f for f in key_for_heatmap if f != 'target_decline']
for i in range(len(feats)):
    for j in range(i+1, len(feats)):
        r = corr_matrix.loc[feats[i], feats[j]]
        if abs(r) > 0.70:
            corr_pairs.append({'Feature_1': feats[i], 'Feature_2': feats[j], 'Correlation': round(r, 3)})
if corr_pairs:
    print(pd.DataFrame(corr_pairs).sort_values('Correlation', ascending=False).to_string(index=False))
else:
    print("No pairs with |r| > 0.70 found.")

print("\nDecline rate by commission size quartile:")
panel['commission_size_quartile'] = pd.qcut(
    panel['commission_l12m'], q=4,
    labels=['Q1 (smallest)', 'Q2', 'Q3', 'Q4 (largest)']
)
gp_analysis = panel.groupby('commission_size_quartile', observed=True).agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
    total_commission_sum=('commission_l12m', 'sum'),
    avg_commission=('commission_l12m', 'mean'),
).assign(
    decline_rate=lambda x: x['decline_count'] / x['total'] * 100,
    pct_of_total_commission=lambda x: (
        x['total_commission_sum'] / x['total_commission_sum'].sum() * 100
    )
)
print(gp_analysis.round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
gp_analysis['decline_rate'].plot(kind='bar', ax=axes[0], color='darkgray')
axes[0].set_title('Decline Rate by Commission Size Quartile', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Decline Rate (%)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].tick_params(axis='x', rotation=30)
gp_analysis['pct_of_total_commission'].plot(kind='bar', ax=axes[1], color='green')
axes[1].set_title('% of Total Commission by Size Quartile', fontsize=11, fontweight='bold')
axes[1].set_ylabel('% of Commission')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].tick_params(axis='x', rotation=30)
plt.suptitle('Commission Size Analysis - Jan 2025', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "jan25_E3_gp_tier_analysis.png", dpi=150)
plt.close()

commission_sorted = panel['commission_l12m'].sort_values().reset_index(drop=True)
cumulative_clients    = (commission_sorted.index + 1) / len(commission_sorted) * 100
cumulative_commission = commission_sorted.cumsum() / commission_sorted.sum() * 100

fig, ax = plt.subplots(figsize=(9, 7))
ax.plot(cumulative_clients, cumulative_commission,
        color='darkgray', linewidth=2.5, label='Actual distribution')
ax.plot([0, 100], [0, 100],
        color='gray', linewidth=1, linestyle='--', label='Perfect equality')

for pct_clients in [50, 75, 90, 99]:
    idx   = int(pct_clients / 100 * len(cumulative_commission)) - 1
    gp_at = cumulative_commission.iloc[idx]
    ax.annotate(
        f'{pct_clients}% of clients\n= {gp_at:.1f}% of commission',
        xy=(pct_clients, gp_at),
        xytext=(pct_clients - 22, gp_at + 8),
        fontsize=8.5,
        arrowprops=dict(arrowstyle='->', color='darkgray', lw=1.2),
        color='darkgray'
    )
    ax.scatter(pct_clients, gp_at, color='red', zorder=5, s=40)

ax.set_title('Commission Concentration - Cumulative Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Cumulative % of Advertiser Groups (sorted smallest to largest)')
ax.set_ylabel('Cumulative % of Total Commission')
ax.legend(fontsize=10)
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "jan25_E4_cumulative_commission_concentration_curve.png", dpi=150)
plt.close()


PART J - JAN 2025 CORRELATION ANALYSIS

Correlation with target_decline (sorted):
fixed_fee_share_mean_l12m            -0.1300
commission_cv_l12m                   -0.1246
commission_peak_concentration_l12m   -0.1152
zero_commission_months               -0.1067
momentum_3m_yoy                      -0.0354
etr_mean_l12m                        -0.0286
total_fixed_gp_l12m                  -0.0280
momentum_6m_yoy                      -0.0258
commission_yoy_growth                -0.0159
tracking_gp_yoy_growth               -0.0158
ctf_gp_l12m                          -0.0046
programme_count_change               -0.0036
commission_l12m                      -0.0006
commission_avg_monthly_l12m          -0.0006
programme_count                       0.0003
transaction_count_l12m                0.0022
core_gp_l12m                          0.0123
tracking_gp_l12m                      0.0162
sas_gp_l12m                           0.0167
etr_yoy_change                        0.0243
active_months_l12m

In [13]:
print("PART K - GP-WEIGHTED HEADLINE SUMMARY")

gp = panel['core_gp_l12m'].clip(lower=0)

unweighted_decline_rate = panel['target_decline'].mean()
gp_weighted_decline_rate = (gp * panel['target_decline']).sum() / gp.sum()

print(f"\nUnweighted decline rate (client count):  {unweighted_decline_rate*100:.1f}%")
print(f"GP-weighted decline rate (core GP):       {gp_weighted_decline_rate*100:.1f}%")
print(f"Gap:                                      {(gp_weighted_decline_rate - unweighted_decline_rate)*100:+.1f} pp")

gp_sorted = panel[['AdvertiserGroup', 'EffectiveBusinessUnit', 'core_gp_l12m', 'target_decline']].copy()
gp_sorted['core_gp_l12m'] = gp_sorted['core_gp_l12m'].clip(lower=0)
gp_sorted = gp_sorted.sort_values('core_gp_l12m', ascending=False).reset_index(drop=True)
gp_sorted['cumulative_gp_pct'] = gp_sorted['core_gp_l12m'].cumsum() / gp_sorted['core_gp_l12m'].sum() * 100
gp_sorted['cumulative_client_pct'] = (gp_sorted.index + 1) / len(gp_sorted) * 100

print(f"\nGP concentration (top clients by core GP, current Jan 2025 snapshot):")
for pct_clients in [1, 5, 10, 25]:
    n = max(1, int(len(gp_sorted) * pct_clients / 100))
    gp_share = gp_sorted.iloc[n - 1]['cumulative_gp_pct']
    print(f"  Top {pct_clients:>2}% of clients ({n:,} groups) = {gp_share:.1f}% of core GP")

print(f"\nDecline rate: top 1% of clients by GP vs the rest:")
top1_n = max(1, int(len(gp_sorted) * 0.01))
top1_uids = set(zip(gp_sorted.head(top1_n)['AdvertiserGroup'], gp_sorted.head(top1_n)['EffectiveBusinessUnit']))
panel['is_top1pct_gp'] = panel.apply(
    lambda r: int((r['AdvertiserGroup'], r['EffectiveBusinessUnit']) in top1_uids), axis=1
)
top1_analysis = panel.groupby('is_top1pct_gp').agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
    total_gp=('core_gp_l12m', 'sum'),
).assign(decline_rate=lambda x: x['decline_count'] / x['total'] * 100)
top1_analysis.index = ['Rest of book', 'Top 1% by GP']
print(top1_analysis.round(1).to_string())

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(gp_sorted['cumulative_client_pct'], gp_sorted['cumulative_gp_pct'],
        color='darkgray', linewidth=2.5, label='Actual GP distribution')
ax.plot([0, 100], [0, 100], color='gray', linewidth=1, linestyle='--', label='Perfect equality')
ax.set_title('Core GP Concentration - Jan 2025 Snapshot', fontsize=13, fontweight='bold')
ax.set_xlabel('Cumulative % of Advertiser Groups (sorted largest GP first)')
ax.set_ylabel('Cumulative % of Core GP')
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "gp_K1_concentration_curve.png", dpi=150)
plt.close()


PART K - GP-WEIGHTED HEADLINE SUMMARY

Unweighted decline rate (client count):  40.6%
GP-weighted decline rate (core GP):       43.8%
Gap:                                      +3.2 pp

GP concentration (top clients by core GP, current Jan 2025 snapshot):
  Top  1% of clients (152 groups) = 40.3% of core GP
  Top  5% of clients (762 groups) = 68.6% of core GP
  Top 10% of clients (1,524 groups) = 81.0% of core GP
  Top 25% of clients (3,811 groups) = 93.7% of core GP

Decline rate: top 1% of clients by GP vs the rest:
              total  decline_count        total_gp  decline_rate
Rest of book  15095     6,123.0000 79,595,072.7000       40.6000
Top 1% by GP    152        68.0000 53,651,352.1000       44.7000


In [14]:
print("PART L - PUBLISHER ENGAGEMENT / FUNNEL-MIX")

has_pub_data = panel['pub_funnel_diversity'] > 0
print(f"\nPublisher data coverage: {has_pub_data.sum():,} of {len(panel):,} "
      f"({has_pub_data.mean()*100:.1f}%)")

print(f"\nL.1 Funnel diversity (number of stages used) vs decline rate:")
div_analysis = panel.groupby('pub_funnel_diversity').agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
    avg_commission=('commission_l12m', 'mean'),
).assign(decline_rate=lambda x: x['decline_count'] / x['total'] * 100)
print(div_analysis.round(1).to_string())

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(div_analysis.index.astype(str), div_analysis['decline_rate'], color='darkgray')
ax.set_title('Decline Rate by Publisher Funnel Diversity (stages used, 0-4)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Funnel Stages Used')
ax.set_ylabel('Decline Rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "publisher_L1_diversity_vs_decline.png", dpi=150)
plt.close()

print("\nL.2 Lower-funnel share vs decline rate:")

def lower_funnel_band(x):
    if pd.isna(x):        return 'No publisher data'
    elif x == 0:           return '0 - None'
    elif x < 0.25:         return '1 - Low (<25%)'
    elif x < 0.50:         return '2 - Medium (25-50%)'
    elif x < 0.75:         return '3 - High (50-75%)'
    else:                  return '4 - Very high (>75%)'

with_pub = panel[has_pub_data].copy()
with_pub['lower_funnel_band'] = with_pub['pub_share_lower_2024'].apply(lower_funnel_band)
lower_analysis = with_pub.groupby('lower_funnel_band').agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
    avg_commission=('commission_l12m', 'mean'),
).assign(decline_rate=lambda x: x['decline_count'] / x['total'] * 100)
print(lower_analysis.round(1).to_string())

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(lower_analysis.index, lower_analysis['decline_rate'], color='red')
ax.set_title('Decline Rate by Lower-Funnel (Cashback/Discount) Commission Share',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Lower-Funnel Share of 2024 Publisher Commission')
ax.set_ylabel('Decline Rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "publisher_L2_lower_funnel_vs_decline.png", dpi=150)
plt.close()

print("\nL.3 Upper-funnel share vs decline rate:")
with_pub['upper_funnel_band'] = with_pub['pub_share_upper_2024'].apply(lower_funnel_band)
upper_analysis = with_pub.groupby('upper_funnel_band').agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
).assign(decline_rate=lambda x: x['decline_count'] / x['total'] * 100)
print(upper_analysis.round(1).to_string())

print("\nL.4 YoY lower-funnel shift vs decline rate:")
with_pub['lower_shift_direction'] = with_pub['pub_lower_funnel_shift_yoy'].apply(
    lambda x: 'Increasing lower-funnel' if x > 0.02
    else ('Decreasing lower-funnel' if x < -0.02 else 'Stable')
)
shift_analysis = with_pub.groupby('lower_shift_direction')['target_decline'].agg(
    total='count', decline='sum'
).assign(decline_rate=lambda x: x['decline'] / x['total'] * 100)
print(shift_analysis.round(1).to_string())

print("\nL.5 GP-weighted decline rate by lower-funnel reliance:")
gp_wp = with_pub['core_gp_l12m'].clip(lower=0)
high_lower = with_pub['pub_share_lower_2024'] >= 0.50
for label, mask in [('High lower-funnel (>=50%)', high_lower), ('Rest', ~high_lower)]:
    sub_gp = gp_wp[mask]
    sub_target = with_pub.loc[mask, 'target_decline']
    rate = (sub_gp * sub_target).sum() / sub_gp.sum() if sub_gp.sum() > 0 else float('nan')
    print(f"  {label}: n={mask.sum():,}, GP-weighted decline rate={rate*100:.1f}%")

PART L - PUBLISHER ENGAGEMENT / FUNNEL-MIX

Publisher data coverage: 7,068 of 15,247 (46.4%)

L.1 Funnel diversity (number of stages used) vs decline rate:
                      total  decline_count  avg_commission  decline_rate
pub_funnel_diversity                                                    
0.0000                 8179     3,335.0000     12,001.8000       40.8000
1.0000                  460       157.0000     47,876.6000       34.1000
2.0000                  576       190.0000      8,121.0000       33.0000
3.0000                 1051       397.0000     18,568.5000       37.8000
4.0000                 4981     2,112.0000    179,020.4000       42.4000

L.2 Lower-funnel share vs decline rate:
                      total  decline_count  avg_commission  decline_rate
lower_funnel_band                                                       
0 - None                607       197.0000     51,959.3000       32.5000
1 - Low (<25%)         1993       846.0000    143,511.4000       42.4000


In [15]:


print("PART M - QUARTERLY YoY MOMENTUM TRAJECTORY (Q1-Q4)")

quarter_cols = [
    'momentum_q1_yoy',
    'momentum_q2_yoy',
    'momentum_q3_yoy',
    'momentum_q4_yoy'
]

# M.1 - Correlation with target by quarter
print("\nM.1 Correlation with target_decline, by quarter:")

quarter_corr = (
    panel[quarter_cols + ['target_decline']]
    .corr()['target_decline']
    .drop('target_decline')
)

print(quarter_corr.to_string())

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(
    ['Q1 (most recent)', 'Q2', 'Q3', 'Q4 (oldest)'],
    quarter_corr.values,
    color='darkgray'
)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Correlation with Decline Target by Quarter', fontsize=13, fontweight='bold')
ax.set_ylabel('Pearson Correlation')

plt.tight_layout()
plt.savefig(OUTPUT_PATH / "jan25_M1_quarterly_momentum_correlation.png", dpi=150)
plt.close()


# M.2 - Decline rate by quarterly momentum band
print("\nM.2 Decline rate by momentum band, per quarter:")

band_labels = [
    'Strong decline (<-20%)',
    'Mild decline (-20% to 0%)',
    'Mild growth (0% to 20%)',
    'Strong growth (>20%)'
]

band_edges = [-1, -0.20, 0, 0.20, 1]

quarter_band_summary = []

for q_col, q_name in zip(quarter_cols, ['Q1', 'Q2', 'Q3', 'Q4']):

    panel['_band'] = pd.cut(
        panel[q_col].clip(-1, 1),
        bins=band_edges,
        labels=band_labels
    )

    for band in band_labels:

        sub = panel[panel['_band'] == band]
        gp = sub['core_gp_l12m'].clip(lower=0)

        quarter_band_summary.append({
            'quarter': q_name,
            'band': band,
            'n': len(sub),
            'unweighted_decline_rate': sub['target_decline'].mean(),
            'gp_weighted_decline_rate': np.average(
                sub['target_decline'],
                weights=gp
            )
        })

panel.drop(columns=['_band'], inplace=True)

quarter_band_df = pd.DataFrame(quarter_band_summary)

print(quarter_band_df.round(3).to_string(index=False))


fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for i, q_name in enumerate(['Q1', 'Q2', 'Q3', 'Q4']):

    sub = quarter_band_df[quarter_band_df['quarter'] == q_name]
    x = np.arange(len(sub))

    axes[i].bar(
        x - 0.2,
        sub['unweighted_decline_rate'],
        width=0.4,
        label='Unweighted',
        color='lightblue'
    )

    axes[i].bar(
        x + 0.2,
        sub['gp_weighted_decline_rate'],
        width=0.4,
        label='GP-weighted',
        color='red'
    )

    axes[i].set_xticks(x)
    axes[i].set_xticklabels(
        sub['band'],
        rotation=25,
        ha='right',
        fontsize=8
    )

    axes[i].set_title(
        f'{q_name} momentum band vs decline rate',
        fontsize=11,
        fontweight='bold'
    )

    axes[i].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    axes[i].legend(fontsize=8)

plt.suptitle(
    'Decline Rate by Momentum Band, Quarter-by-Quarter - Jan 2025',
    fontsize=13,
    fontweight='bold'
)

plt.tight_layout()
plt.savefig(OUTPUT_PATH / "jan25_M2_quarterly_bands_vs_decline.png", dpi=150)
plt.close()


# M.3 - Decline rate by trajectory shape
print("\nM.3 Trajectory shape vs decline rate:")

q1 = panel['momentum_q1_yoy']
yoy_12m = panel['commission_yoy_growth']

worsening = (
    (q1 < 0) &
    (yoy_12m < 0) &
    (q1 < yoy_12m)
)

reversal = (
    (q1 < 0) &
    (yoy_12m >= 0)
)

panel['_trajectory'] = np.select(
    [worsening, reversal],
    [
        'Worsening (declining, getting worse)',
        'Reversal (just turned negative)'
    ],
    default='Neither'
)

traj_summary = panel.groupby('_trajectory').agg(
    n=('target_decline', 'count'),
    unweighted_decline_rate=('target_decline', 'mean'),
    total_gp=('core_gp_l12m', lambda x: x.clip(lower=0).sum())
)

traj_summary['gp_weighted_decline_rate'] = (
    panel.groupby('_trajectory')
    .apply(
        lambda g: np.average(
            g['target_decline'],
            weights=g['core_gp_l12m'].clip(lower=0)
        )
    )
)

panel.drop(columns=['_trajectory'], inplace=True)

print(traj_summary.round(3).to_string())

PART M - QUARTERLY YoY MOMENTUM TRAJECTORY (Q1-Q4)

M.1 Correlation with target_decline, by quarter:
momentum_q1_yoy   -0.0354
momentum_q2_yoy   -0.0196
momentum_q3_yoy   -0.0047
momentum_q4_yoy   -0.0047

M.2 Decline rate by momentum band, per quarter:
quarter                      band    n  unweighted_decline_rate  gp_weighted_decline_rate
     Q1    Strong decline (<-20%) 4011                   0.6140                    0.6660
     Q1 Mild decline (-20% to 0%) 1050                   0.4950                    0.4710
     Q1   Mild growth (0% to 20%)  885                   0.4510                    0.3850
     Q1      Strong growth (>20%) 3760                   0.3810                    0.3540
     Q2    Strong decline (<-20%) 3942                   0.5630                    0.5550
     Q2 Mild decline (-20% to 0%) 1014                   0.4910                    0.4370
     Q2   Mild growth (0% to 20%)  806                   0.4780                    0.3990
     Q2      Strong growth

In [16]:
print("PART N - ACCELERATION BETWEEN QUARTERS")

panel['accel_q2_to_q1'] = panel['momentum_q1_yoy'] - panel['momentum_q2_yoy']
panel['accel_q3_to_q2'] = panel['momentum_q2_yoy'] - panel['momentum_q3_yoy']
panel['accel_q4_to_q3'] = panel['momentum_q3_yoy'] - panel['momentum_q4_yoy']
accel_cols = ['accel_q2_to_q1', 'accel_q3_to_q2', 'accel_q4_to_q3']

# N.1 - Correlation with target by acceleration pair
print(f"\nN.1 Correlation with target_decline, by acceleration pair:")
accel_corr = panel[accel_cols + ['target_decline']].corr()['target_decline'].drop('target_decline')
print(accel_corr.to_string())

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(['Q2->Q1\n(most recent)', 'Q3->Q2', 'Q4->Q3\n(oldest)'], accel_corr.values, color='darkgray')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Correlation with Decline Target by Acceleration Pair', fontsize=13, fontweight='bold')
ax.set_ylabel('Pearson Correlation')
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "jan25_N1_acceleration_correlation.png", dpi=150)
plt.close()

# Compare acceleration direction within the same current momentum band.
print("\nN.2 Decline rate by acceleration direction within Q1 momentum band:")
band_labels = ['Strong decline (<-20%)', 'Mild decline (-20% to 0%)',
               'Mild growth (0% to 20%)', 'Strong growth (>20%)']
band_edges  = [-1, -0.20, 0, 0.20, 1]
panel['_q1_band'] = pd.cut(panel['momentum_q1_yoy'].clip(-1, 1), bins=band_edges, labels=band_labels)
panel['_accel_direction'] = np.where(
    panel['accel_q2_to_q1'] < 0, 'Accelerating decline (worse than Q2)',
    'Decelerating / improving vs Q2'
)

interaction_rows = []
for band in band_labels:
    for direction in ['Accelerating decline (worse than Q2)', 'Decelerating / improving vs Q2']:
        sub = panel[(panel['_q1_band'] == band) & (panel['_accel_direction'] == direction)]
        if len(sub) == 0:
            continue
        gp_sub = sub['core_gp_l12m'].clip(lower=0)
        gp_weighted_rate = (
            (gp_sub * sub['target_decline']).sum() / gp_sub.sum() if gp_sub.sum() > 0 else np.nan
        )
        interaction_rows.append({
            'q1_band': band, 'direction': direction, 'n': len(sub),
            'unweighted_decline_rate': sub['target_decline'].mean(),
            'gp_weighted_decline_rate': gp_weighted_rate,
        })
interaction_df = pd.DataFrame(interaction_rows)
print(interaction_df.round(3).to_string(index=False))

# Difference in decline rate within each Q1 band
print("\nGap in decline rate within each Q1 band:")
pivot_uw = interaction_df.pivot(index='q1_band', columns='direction', values='unweighted_decline_rate')
pivot_uw = pivot_uw.reindex(band_labels)
gap = pivot_uw['Accelerating decline (worse than Q2)'] - pivot_uw['Decelerating / improving vs Q2']
print(gap.round(3).to_string())

panel.drop(columns=['_q1_band', '_accel_direction'], inplace=True)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(band_labels))
width = 0.35
accel_rates = pivot_uw['Accelerating decline (worse than Q2)'].values
decel_rates = pivot_uw['Decelerating / improving vs Q2'].values
ax.bar(x - width/2, accel_rates, width, label='Accelerating decline', color='red')
ax.bar(x + width/2, decel_rates, width, label='Decelerating / improving', color='lightblue')
ax.set_xticks(x)
ax.set_xticklabels(band_labels, rotation=20, ha='right', fontsize=9)
ax.set_title('Decline Rate by Q1 Momentum Band, Split by Acceleration Direction', fontsize=13, fontweight='bold')
ax.set_ylabel('Decline Rate')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "jan25_N2_acceleration_within_q1_band.png", dpi=150)
plt.close()


PART N - ACCELERATION BETWEEN QUARTERS

N.1 Correlation with target_decline, by acceleration pair:
accel_q2_to_q1   0.0027
accel_q3_to_q2   0.0038
accel_q4_to_q3   0.0119

N.2 Decline rate by acceleration direction within Q1 momentum band:
                  q1_band                            direction    n  unweighted_decline_rate  gp_weighted_decline_rate
   Strong decline (<-20%) Accelerating decline (worse than Q2) 2561                   0.6550                    0.7300
   Strong decline (<-20%)       Decelerating / improving vs Q2 1450                   0.5400                    0.5070
Mild decline (-20% to 0%) Accelerating decline (worse than Q2)  524                   0.5320                    0.4220
Mild decline (-20% to 0%)       Decelerating / improving vs Q2  526                   0.4580                    0.5520
  Mild growth (0% to 20%) Accelerating decline (worse than Q2)  377                   0.4750                    0.4640
  Mild growth (0% to 20%)       Decelerating /

In [17]:
print("PART O - COMMISSION CONCENTRATION (CUMULATIVE COMMISSION CONCENTRATION CURVE)")

comm_sorted = panel[['AdvertiserGroup', 'EffectiveBusinessUnit', 'commission_l12m']].copy()
comm_sorted['commission_l12m'] = comm_sorted['commission_l12m'].clip(lower=0)
comm_sorted = comm_sorted.sort_values('commission_l12m', ascending=False).reset_index(drop=True)
comm_sorted['cumulative_commission_pct'] = (
    comm_sorted['commission_l12m'].cumsum() / comm_sorted['commission_l12m'].sum() * 100
)
comm_sorted['cumulative_client_pct'] = (comm_sorted.index + 1) / len(comm_sorted) * 100

print("\nCommission concentration - Jan 2025:")
for pct_clients in [1, 5, 10, 25]:
    n = max(1, int(len(comm_sorted) * pct_clients / 100))
    comm_share = comm_sorted.iloc[n - 1]['cumulative_commission_pct']
    print(f"  Top {pct_clients:>2}% of clients ({n:,} groups) = {comm_share:.1f}% of trailing commission")

# Gini coefficient 
x = panel['commission_l12m'].clip(lower=0).sort_values().to_numpy()
n = len(x)
cum_x = x.cumsum()
gini = (2 * (np.arange(1, n + 1) * x).sum() - (n + 1) * cum_x[-1]) / (n * cum_x[-1])
print(f"\nGini coefficient (trailing commission, Jan 2025 snapshot): {gini:.4f}")


fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(comm_sorted['cumulative_client_pct'], comm_sorted['cumulative_commission_pct'],
        color='darkgray', linewidth=2.5, label='Actual commission distribution')
ax.plot([0, 100], [0, 100], color='gray', linewidth=1, linestyle='--', label='Perfect equality')
ax.set_title(f'Commission Concentration - Jan 2025 Snapshot (Gini = {gini:.3f})',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Cumulative % of Advertiser Groups (sorted largest commission first)')
ax.set_ylabel('Cumulative % of Trailing Commission')
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "commission_O1_cumulative_commission_concentration_curve.png", dpi=150)
plt.close()



PART O - COMMISSION CONCENTRATION (CUMULATIVE COMMISSION CONCENTRATION CURVE)

Commission concentration - Jan 2025:
  Top  1% of clients (152 groups) = 62.3% of trailing commission
  Top  5% of clients (762 groups) = 83.5% of trailing commission
  Top 10% of clients (1,524 groups) = 90.8% of trailing commission
  Top 25% of clients (3,811 groups) = 97.4% of trailing commission

Gini coefficient (trailing commission, Jan 2025 snapshot): 0.9362
